<a href="https://colab.research.google.com/github/vivianesilper/cienciadedados/blob/Processamento-de-Linguagem-Natural/SupervisionandoRegras_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Analise de Sentimento Baseado em Regras

Nltk

In [ ]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
mas = SentimentIntensityAnalyzer()

In [ ]:
a = "I hate this movie but my husband to appreciate to movie in the die. I don't to agree with his"
x = mas.polarity_scores(a)
print(x)

{'neg': 0.371, 'neu': 0.502, 'pos': 0.127, 'compound': -0.7792}


In [ ]:
b = ':('
c = mas.polarity_scores(b)
print(c)


{'neg': 1.0, 'neu': 0.0, 'pos': 0.0, 'compound': -0.4404}


In [ ]:
print(type([a,b]))

<class 'list'>


In [ ]:
print(type(a))

<class 'str'>


In [ ]:
print(c['compound'])

-0.4404


In [ ]:
from textblob import TextBlob
teste = TextBlob("I love NY!")
print(teste.sentiment)

Sentiment(polarity=0.625, subjectivity=0.6)


In [ ]:
!pip install translate

In [ ]:
from translate import Translator
#translator = Translator(from_lang="pt", to_lang="es")
translator = Translator(from_lang="pt", to_lang="en")
tradução = translator.translate("Vc está chateaado com algo me parece descontente")
print(tradução)

You are upset about something that seems displeased to me


In [ ]:
t = mas.polarity_scores(tradução)
print(t)

{'neg': 0.407, 'neu': 0.593, 'pos': 0.0, 'compound': -0.6705}



Comparar VADER com LSTM

Suoervisionado com LSTM

Regras com o VADER

**VADER**

Vamos criar um coluna vader_sentiment

Vamos registrar o Polarity Score nesta coluna

Vamos remover o score para Compound

Vamos buscar o maior escore entre Positivo, Negativo e Neutro

Registrar na coluna vader_sentiment criada
Avaliar a performance comparado com a coluna sentiment

In [ ]:
import pandas as pd
from google.colab import files
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score

from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
#from keras.utils import np_utils

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [ ]:
files.upload()

NameError: name 'files' is not defined

In [ ]:
tweets = pd.read_csv("Tweets2.csv")
tweets

NameError: name 'pd' is not defined

In [ ]:
tweets.groupby(['sentiment']).size()

sentiment
Irrelevant    12990
Negative      22542
Neutral       18318
Positive      20832
dtype: int64

In [ ]:
tweets.loc[tweets['sentiment']== 'Irrelevante','sentiment'] = 'Neutral'

In [ ]:
tweets = tweets.dropna(subset=['text'])
tweets.reset_index(drop=True, inplace=True)

In [ ]:
tweets.shape

(73996, 4)

In [ ]:
token = Tokenizer(num_words=100)
token.fit_on_texts(tweets['text'].values)

**Supervisionado**

In [ ]:
!pip install keras.utils

  Preparing metadata (setup.py) ... done
  Created wheel for keras.utils: filename=keras_utils-1.0.13-py3-none-any.whl size=2631 sha256=a7ac7c7f251ba65e3892b5cf56a9c1969fcb139fda698818f58eb21373dac357
  Stored in directory: /root/.cache/pip/wheels/5c/c0/b3/0c332de4fd71f3733ea6d61697464b7ae4b2b5ff0300e6ca7a
Successfully built keras.utils


In [ ]:
from tensorflow import keras


In [ ]:
token = Tokenizer(num_words=100)
token.fit_on_texts(tweets['text'].values)

In [ ]:
X = token.texts_to_sequences(tweets['text'].values)
X = pad_sequences(X, padding="post", maxlen=100)

In [ ]:
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(tweets['text'].values)
print(y)

[64619 27233 64618 ... 36810 36803 36735]


In [ ]:
from tensorflow.keras.utils import to_categorical

In [ ]:
#!pip install kera.utils
#import pandas as pd
#from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import LabelEncoder
#from sklearn.feature_extraction.text import CountVectorizer
#from sklearn.metrics import confusion_matrix
#from keras.models import Sequential
#from keras.layers import Dense, Dropout, Flatten, Embedding
#from google.colab import files

# prompt: keras.models

#from tensorflow import keras
#!pip install keras.utils (necessário para importar o numpy_utils), sem esse não a possibilidade de uar o método to_categorical.

#import pandas as pd
#from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import LabelEncoder
#from keras.models import Sequential
#from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
#from keras.preprocessing.text import Tokenizer
#from keras.preprocessing.sequence import pad_sequences
#from keras.utils import np_utils
#from google.colab import files
#import numpy as np
#import numpy as np_utils

#import numpy as np
#from keras.utils import to_categorical

# Convert the labels to categorical format
#y = to_categorical(y)

# Print the categorical labels
#print(y)

AttributeError: 'function' object has no attribute 'to_categorical'

In [ ]:
print(X)

[[13  4  2 ...  0  0  0]
 [ 2  3  1 ...  0  0  0]
 [13  4  2 ...  0  0  0]
 ...
 [23  1  6 ...  0  0  0]
 [23  1  6 ...  0  0  0]
 [23 30  1 ...  0  0  0]]


In [ ]:
modelo = Sequential()
modelo.add(Embedding(input_dim= len(token.word_index), output_dim=128, input_length=X.shape[1]))
modelo.add(SpatialDropout1D(0.2))
modelo.add(LSTM(units=196, dropout=0.2, recurrent_dropout=0, activation='tanh',
                recurrent_activation='sigmoid', unroll=False, use_bias=True))
modelo.add(Dense(units=3,activation="softmax"))

In [ ]:
modelo.compile(loss='categorical_crossentropy', optimizer='adam',metrics = ['acurracy'])
print(modelo.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          4324224   
                                                                 
 spatial_dropout1d (Spatial  (None, 100, 128)          0         
 Dropout1D)                                                      
                                                                 
 lstm (LSTM)                 (None, 196)               254800    
                                                                 
 dense (Dense)               (None, 3)                 591       
                                                                 
Total params: 4579615 (17.47 MB)
Trainable params: 4579615 (17.47 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
#modelo.fit(X_train, y_train, epochs=10, batch_size=30,verbose=True,validation_data=(X_test, y_test))

NameError: name 'modelo' is not defined

In [ ]:
#modelo.fit(X_train, y_train, epochs=10, batch_size=30,verbose=True)

In [ ]:
#from pandas.core.internals.managers import T
#mas = SentimentIntensityAnalyzer()
#Tweets['vander_sentiment'] = ''

#for y in range(len(tweets.index)):
#   x = mas.polarity(tweets['text].iloc[y])
#del x['compound']
#maior = max(x,key=x.get)
#tweets.loc[y,'vander_sentiment'] = maior

In [ ]:
#tweets.groupby(['vander_sentiment']).size()

KeyError: 'vander_sentiment'

In [ ]:
#tweets.groupby(['sentiment]).size()

In [ ]:
#tweets.loc[tweets['vander_sentiment']== 'neu', 'vander_sentiment'] = 'Neutral'
#tweets.loc[tweets['vander_sentiment']== 'neg', 'vander_sentiment'] = 'Negative'
#tweets.loc[tweets['vander_sentiment']== 'neu', 'vander_sentiment'] = 'Positive'

KeyError: 'vander_sentiment'

In [ ]:
#tweets.groupby(['vander_sentiment]).size()

In [ ]:
#_pred = tweets['vander_sentiment']
#y_test = tweets['sentiment']
#cm = confusion_matrix(y_test, y_pred)
#rint(cm)

KeyError: 'vander_sentiment'

In [ ]:
#accuracy = accuracy_score(y_test, y_pred)
#print(accuracy)

NameError: name 'y_test' is not defined


#algoritimos NLP

Introdução ao Transformers
> worvec2, glove, RNN, Seq2seq, transform, bert, t5, GPT-3






Aplicação de Pergutas e Respostas

In [ ]:
!pip install transformers
from transformers import pipeline


In [ ]:
qea = pipeline("question-answering", model="pierreguillou/bert-base-cased-squad-v1.1-portuguese") #classificando o grua de acuracia do modelo.

config.json:   0%|          | 0.00/862 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
texto = "Carl Sagan foi um cientista norte-americano. Sagan é autor demais de 600 publicações cientificas e também de mais de vinte livros de ciência e física"
#pergunta = "Quantos livros Carl Sagan tem publicado? "
#pergunta = "Quem é Carl Sagan?"
pergunta = "Quantas publicações cientificas CArl Sagan tem publicado?"
resposta = qea(question=pergunta, context=texto)
print("Pergunta: ", pergunta)
print("Resposta: ", resposta['answer'])
print("Score: ", resposta['score'])

Pergunta:  Quantas publicações cientificas CArl Sagan tem publicado?
Resposta:  600
Score:  0.8944277167320251
